In [1]:
import numpy as np
from pylab import gca
import numpy as np
import math
from tqdm import tqdm
import torch
import torchvision
from diffusionsr.datasets.dataset import SimulationXZDataset
import time 
import time 
import torch.nn as nn
from torch.utils import data
import torch.nn.functional as F
from torchvision import transforms
from torchvision.utils import save_image
from torchvision.datasets import MNIST
import torchvision.transforms.functional as TF
from torch.optim import lr_scheduler
import time
import os

from skimage.metrics import structural_similarity as ssim_id
import numpy as np
import cv2
import os
from torchvision import transforms
from torch.utils.data import DataLoader
from pathlib import Path
from torch.utils.data import Dataset
import pdb
from PIL import Image
import matplotlib.pyplot as plt 
# from datasets.dataset import TemperatureXZDataset
from diffusionsr.runners.train_diffusion import forwardpass
from diffusionsr.analysis.analysis_functions import predict_lrenc, predict_mobilenet, plot_images, get_profile, load_mobilenet, load_encoder, load_diffusion, PSNR, SSIM, predict_modified_diffusion

/home/shohom-tfc/miniconda3/envs/LPBFDiffusion/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def frame_tick(frame_width = 2, tick_width = 1.5):
    ax = gca()
    for axis in ['top','bottom','left','right']:
        ax.spines[axis].set_linewidth(frame_width)
    plt.tick_params(direction = 'in', 
                    width = tick_width)
def legend(location = 'best', fontsize = 8):
        plt.legend(loc = location, fontsize = fontsize, frameon = False)

In [3]:

device = 'cuda'


def compute_alpha(beta, t):
    beta = torch.cat([torch.zeros(1).to(beta.device), beta], dim=0).to(device)
    # print(beta.device, t.device)
    a = (1 - beta).cumprod(dim=0).index_select(0, t + 1).view(-1, 1, 1, 1)
    return a



def predict_modified_ddim_diffusion(model, lr_enc, res, hr, lr, upscaled_lr, encoding, dataset, seq, timesteps = 200, skip = 1, schedule = 'linear', **kwargs):
    
    # skip =timesteps // self.args.timesteps
    seq = range(0, timesteps, skip)
    
    def cosine_beta_schedule(timesteps, s=0.008):

        steps = timesteps + 1
        x = torch.linspace(0, timesteps, steps)
        alphas_cumprod = torch.cos(
            ((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0.0001, 0.9999)

    def linear_beta_schedule(timesteps):
        beta_start = 0.0001
        beta_end = 0.02
        return torch.linspace(beta_start, beta_end, timesteps)


    def quadratic_beta_schedule(timesteps):
        beta_start = 0.0001
        beta_end = 0.02
        return torch.linspace(beta_start**0.5, beta_end**0.5, timesteps) ** 2

    def sigmoid_beta_schedule(timesteps):
        beta_start = 0.0001
        beta_end = 0.02
        betas = torch.linspace(-6, 6, timesteps)
        return torch.sigmoid(betas) * (beta_end - beta_start) + beta_start
    if schedule == 'linear':
        betas = linear_beta_schedule(timesteps=timesteps)
    elif schedule == 'quadratic':
        betas = quadratic_beta_schedule(timesteps=timesteps)
    elif schedule == 'cosine':
        betas = cosine_beta_schedule(timesteps=timesteps)
    elif schedule == 'sigmoid':
        betas = sigmoid_beta_schedule(timesteps=timesteps)
    b = betas

    if len(lr.shape) < 4:
        img = (lr.view(lr.shape[0], 1, lr.shape[1], lr.shape[2]).to(device))
        target = (hr.view(hr.shape[0], 1, hr.shape[1], hr.shape[2]).to(device))
    else:
        img = lr.to(device)
        target = hr.to(device)
    if encoding:
            # x_e = forwardpass(lr_enc, lr.to(device).float(), factor = train_dataset.factor)
        if len(lr.shape)< 4:
            x_e = forwardpass(lr_enc, lr.view(lr.shape[0],1, lr.shape[1], lr.shape[2]).to(device).float())
        else:
            x_e = forwardpass(lr_enc, lr.to(device).float())
    else:
        x_e = upscaled_lr.to(device).float().repeat(1,1,1, 1)
    shape=hr.shape
    with torch.no_grad():
        x = torch.randn(shape, device=device)
        n = x.size(0)
        seq_next = [-1] + list(seq[:-1])
        x0_preds = []
        xs = [x]
        for i, j in zip(reversed(seq), reversed(seq_next)):
            t = (torch.ones(n) * i).to(x.device)
            next_t = (torch.ones(n) * j).to(x.device)
            at = compute_alpha(b, t.long())
            at_next = compute_alpha(b, next_t.long())
            xt = xs[-1].to('cuda')
            et = model(xt, t, x_e)
            x0_t = (xt - et * (1 - at).sqrt()) / at.sqrt()
            x0_preds.append(x0_t.to('cpu'))
            c1 = (
                kwargs.get("eta", 0) * ((1 - at / at_next) * (1 - at_next) / (1 - at)).sqrt()
            )
            c2 = ((1 - at_next) - c1 ** 2).sqrt()
            xt_next = at_next.sqrt() * x0_t + c1 * torch.randn_like(x) + c2 * et
            xs.append(xt_next.to('cpu'))
    result = dataset.unscale_data(xs[-1], input_type = 'hr') 
    return dataset.unscale_data(lr, input_type='lr'), result, dataset.unscale_data(target.cpu(), input_type = 'hr'), xs, b





In [4]:
class Simulation():
    def __init__(self, folder, diff_steps, schedule):
        self.folder = folder
        self.diff_steps = diff_steps
        self.schedule = schedule
        self.kp = None
        self.profile = None
        self.max_depth = None
        self.kp_depth = None
    def get_loss_curve(self):
        test_loss = np.loadtxt(os.path.join(self.folder, 'validation_loss_epoch.txt'))
        train_loss = np.loadtxt(os.path.join(self.folder, 'loss_epoch.txt'))
        return train_loss, test_loss


In [5]:
# diffusion_results_dir = './runs/direct/diffusionimplicitencoded/2023_02_26_02_05_27/standardize/n_steps_1'
# encoder_results_dir = './runs/direct/encoder/2023_02_26_02_05_27/standardize/n_steps_1'
diffusion_results_dir = './runs/direct/diffusionimplicitupscaled/2026_04_27_17_22_41/standardize/n_steps_1'
encoder_results_dir = './runs/direct/encoderupscaled/2026_04_27_17_22_41/standardize/n_steps_1'
timesteps = 1000
conditioning = 'implicit'
encoding = 'True'
schedule = 'linear'
device= 'cuda'
encode_bool = encoding == 'True'

os.environ['CUDA_VISIBLE_DEVICES']  = "0"
batch_size = 1
downscale_method = 'direct'
analysis_folder = f'analyzed_figures_paper_3_20/{timesteps}_{conditioning}_{schedule}_{downscale_method}'
os.makedirs( analysis_folder, exist_ok = True)

data_folder = './data/simulation_basis_v3_laser_velocity_xz_cross_section_data'
train_dataset = SimulationXZDataset(downscale_method = 'direct', split = 'train', root_folder = data_folder , return_info = True)
test_dataset = SimulationXZDataset(downscale_method ='direct', split = 'test', root_folder = data_folder, return_info = True)
dev_dataset = SimulationXZDataset(downscale_method = 'direct', split = 'dev', root_folder = data_folder, return_info = True)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
dev_dataloader = DataLoader(dev_dataset, batch_size=1, shuffle=False, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=True)


Using normalize method: ... standardize
Using all 1 fields
Processing dataset with 1 fields


FileNotFoundError: No files found in specified directory: ./data/simulation_basis_v3_laser_velocity_xz_cross_section_data/train/LR/direct/1x/

In [ ]:
from diffusionsr.runners.train_diffusion import DiffusionModel

def initialize_diffusion(diff_dir, enc_dir, datasets, timesteps, conditioning, encoding, schedule, device):
    ''' 
    Parameters:
        diff_dir: (str) Diffusion results directory
        enc_dir: (str) Encoder results directory
        datasets: (tuple or list) The three dataset objects corresponding to the train/validation/test splits, in the order (train, validation, test).
        timesteps: (int) The number of timesteps used for the diffusion model during training
        conditioning: (boolean) If true, the diffusion model assumes the LR passes through an encoder before being used for conditioning
        schedule: (str) Variance schedule used for training the diffusion model
        device: (str) 'cuda' for GPU, 'cpu' for CPU.
    Returns:
        diffusion_model: Custom DiffusionModel object that enables sampling with either DDIM or DDPM samplers.
    '''

    diffusion_model = DiffusionModel(results_folder=diff_dir,
                                    lr_encoder_folder=enc_dir,
                                    train_dataset=datasets[0],
                                    dev_dataset=datasets[1],
                                    test_dataset=datasets[2],
                                    timesteps=timesteps,
                                    conditioning=conditioning,
                                    encoding=encoding,
                                    schedule=schedule,
                                    device=device,enc_output = False
                                    )
    diffusion_model.load_saved_model()
    return diffusion_model

diff_model = initialize_diffusion(diff_dir=diffusion_results_dir,
                                  enc_dir=encoder_results_dir,
                                  datasets=[train_dataset,
                                            dev_dataset, test_dataset],
                                  timesteps=timesteps,
                                  conditioning=conditioning,
                                  encoding=encoding,
                                  schedule=schedule,
                                  device=device)


lr_enc = load_encoder(encoder_results_dir, dataset = train_dataset)


In [ ]:
def predict_refactored_diffusion(diff_model, lr_enc, res, hr, lr, upscaled_lr, dataset, skip = 50):

    '''
    Return the predictions for the Diffusion model given an input batch
    Parameters:
    diff_model: DiffusionModel object
    lr_enc: Torch network module object, representing the trained RRDN encoder model
    res: Torch tensor, Residual between HR and LR data
    hr: Torch tensor, High Resolution data
    lr: Torch tensor, Low Resolution data
    upscaled_lr: Torch tensor, Bicubic upscaled low resolution data
    dataset: Dataset object, used for rescaling data
    Returns:
    Low resolution data, scaled to original space, 4-D numpy array (batch, channels, height, width)
    Output (Super-resolution), scaled to original space, 4-D numpy array
    High resolution data, scaled to original space, 4-D numpy array 
    '''

    if len(lr.shape) < 4:
        img = (lr.view(lr.shape[0], 1, lr.shape[1], lr.shape[2]).to(device))
        target = (hr.view(hr.shape[0], 1, hr.shape[1], hr.shape[2]).to(device))
    else:
        img = lr.to(device)
        target = hr.to(device)
    if len(lr.shape) < 4:

        x_e = forwardpass(lr_enc, lr.view(lr.shape[0],1, lr.shape[1], lr.shape[2]).to(device).float(), factor = dataset.factor)
    else:
        x_e = forwardpass(lr_enc, lr.to(device).float(), factor = dataset.factor)
        x_e = forwardpass(lr_enc, lr.to(device).float())
        all_images = diff_model.batch_sample(dataset = train_dataset, batch = hr, x_e = x_e, sampler = 'DDIM', skip= skip)                
        result = dataset.unscale_data(all_images.cpu().numpy()[-1, 0], input_type = 'residual') + dataset.unscale_data(upscaled_lr.numpy(), input_type = 'upscaled_lr')
        
    return dataset.unscale_data(lr, input_type='lr'), result, dataset.unscale_data(target.cpu(), input_type = 'hr')
skip = 30
for batch_idx, (res, hr, lr, upscaled_lr, info) in enumerate(train_dataloader):            
    #input, result_diffusion, target = predict_refactored_diffusion(diff_model, lr_enc, res, hr, lr, upscaled_lr, train_dataloader.dataset)
    input, result_diffusion, target, _, _ = predict_modified_ddim_diffusion(diff_model.model, lr_enc, res, hr, lr, upscaled_lr,encoding = encode_bool,dataset =  train_dataset,seq= None, timesteps = timesteps,skip = skip, schedule = 'linear')
    break


In [ ]:
last_power = ''
modeltype = 'Diffusion' # Defines part of the title of the image

diff_results_global = {}
skip = 50
for trial_key in range(10):
    enc_depths_sim = []
    diff_depths_sim = []
    gt_depths_sim = []
    enc_kp_sim = []
    diff_kp_sim = []
    gt_kp_sim = []
    gt_results = {}
    enc_results = {}
    diff_results = {}
    for batch_idx, (res, hr, lr, upscaled_lr, info) in tqdm(enumerate(test_dataloader), total = len(test_dataloader)):  
    

        oldtime = time.time()

        input, result, target = predict_lrenc(lr_enc,res, hr, lr, upscaled_lr, train_dataset)
        input, result_diffusion, target, _, _ = predict_modified_ddim_diffusion(diff_model.model, lr_enc, res, hr, lr, upscaled_lr,encoding = encode_bool,dataset =  train_dataset,seq= None, timesteps = timesteps,skip = skip, schedule = 'linear')
        # input, result_diffusion, target = predict_refactored_diffusion(diff_model, lr_enc, res, hr, lr, upscaled_lr, dev_dataloader.dataset)
        # input, result_diffusion, target = predict_modified_diffusion(diff_model.model, lr_enc, res, hr, lr, upscaled_lr,dataset =  train_dataset, timesteps = timesteps, schedule = 'linear')
        
        info = info[0] # info is technically returned with an extra dimension, this command removes it.
        time_elapsed = time.time()-oldtime
        # print(last_power)
        if last_power != 'power' + str(info[0].item()) + 'vel' + str(info[1].item()):
            
            timestep = 0

            enc_depths_sim = []
            diff_depths_sim = []
            gt_depths_sim = []

            enc_kp_sim = []
            diff_kp_sim = []
            gt_kp_sim = []

        else:
            timestep += 1
        profile_diff, kp_diff = get_profile(result_diffusion[0])
        profile_gt, kp_gt = get_profile(target[0])
        profile_enc, kp_enc = get_profile(result[0])


        enc_kp_sim.append(5*(80-np.min(kp_enc)))
        gt_kp_sim.append(5*(80-np.min(kp_gt)))
        diff_kp_sim.append(5*(80-np.min(kp_diff)))

        enc_depths_sim.append(5*(80-np.min(profile_enc)))
        gt_depths_sim.append(5*(80-np.min(profile_gt)))
        diff_depths_sim.append(5*(80-np.min(profile_diff)))

        last_power = 'power' + str(info[0].item()) + 'vel' + str(info[1].item())  
        gt_results['key_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = gt_kp_sim
        enc_results['key_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = enc_kp_sim
        diff_results['key_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = diff_kp_sim
        gt_results['depth_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = gt_depths_sim
        enc_results['depth_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = enc_depths_sim
        diff_results['depth_power' + str(info[0].item()) + 'vel' + str(info[1].item()) ] = diff_depths_sim
    diff_results_global[trial_key] = diff_results
        # if batch_idx > 3:
        #     break
        # break

In [ ]:
np.fft.fftfreq(np.array(signal).size, d = time_signal[1] - time_signal[0])

In [ ]:
def calc_fft_freq(signal, timestep = 5e-6):
    fft = np.fft.rfft(signal - np.mean(signal))
    power_spectrum = np.abs(fft)**2
    time_signal = np.arange(len(signal))*timestep
    freq = np.fft.rfftfreq(np.array(signal).size, d = time_signal[1] - time_signal[0])
    dominant_frequency = freq[np.argmax(power_spectrum)]
    return dominant_frequency, freq, power_spectrum
for key in diff_results.keys():
    if 'key' in key:
        print(key)



        gt_dominant_frequency, gt_freq, gt_power_spectrum = calc_fft_freq(gt_results[key][30:90])
        diff_dominant_frequency, diff_freq, diff_power_spectrum = calc_fft_freq(diff_results[key][30:90])
        enc_dominant_frequency, enc_freq, enc_power_spectrum = calc_fft_freq(enc_results[key][30:90])

        # plt.plot(gt_results[key][30:80])
        # plt.plot(diff_results[key][30:80])
        # plt.plot(enc_results[key][30:80])
        # plt.show()
        plt.title('{}, {},{}'.format(gt_dominant_frequency,diff_dominant_frequency, enc_dominant_frequency ))
        gt_amp = np.sqrt(np.mean((gt_results[key][30:80] - np.mean(gt_results[key][30:80]))**2))
        diff_amp = np.sqrt(np.mean((diff_results[key][30:80] - np.mean(diff_results[key][30:80]))**2))

        # plt.title('{}, {}'.format(np.sqrt(np.mean(gt_results[key][30:80])),np.sqrt(np.mean(diff_results[key][30:80])) ))
        # plt.title('{},{}'.format(gt_amp,diff_amp))
        plt.plot(diff_freq, diff_power_spectrum, label = "Diffusion")
        
        plt.plot(gt_freq, gt_power_spectrum, label = "Ground Truth")
        plt.plot(enc_freq, enc_power_spectrum, label = 'CNN')
        plt.legend()
        # plt.show()

        # plt.title(diff_freq[np.argmax(diff_power_spectrum)])

        plt.show()


In [ ]:
diff_results_global[trial_key].keys()

In [ ]:
parameters = list(diff_results_global[trial_key].keys())[0].split('key_power')[-1].split('vel')

In [ ]:

diff_amps_global = []
for trial_key in diff_results_global.keys():
    diff_results = diff_results_global[trial_key]
    enc_amps = []
    gt_amps = []
    diff_amps = []
    powers = []
    velocities = []
    for key in diff_results_global[trial_key].keys():
        print(key, gt_results.keys())
    
        if 'key' in key:
            # plt.figure(dpi = 150, figsize = np.array([4,3])*1.3)
            # plt.plot(np.arange(len(diff_results[key]))*5, diff_results[key], '--')
            # plt.plot(np.arange(len(diff_results[key]))*5, gt_results[key], 'k-')
            # plt.plot(np.arange(len(diff_results[key]))*5, enc_results[key], '--')
            gt_amp = np.sqrt(np.mean((gt_results[key][30:80] - np.mean(gt_results[key][30:80]))**2))
            diff_amp = np.sqrt(np.mean((diff_results[key][30:80] - np.mean(diff_results[key][30:80]))**2))
            enc_amp  = np.sqrt(np.mean((enc_results[key][30:80] - np.mean(enc_results[key][30:80]))**2))
            gt_amps.append(gt_amp)
            enc_amps.append(enc_amp)
            diff_amps.append(diff_amp)
            print(len(diff_amps))
            parameters = key.split('key_power')[-1].split('vel')
            powers.append(float(parameters[0]))
            velocities.append(float(parameters[1]))
            # frame_tick()
            # legend()
            # plt.xlim([0, 500])
            # plt.ylim([0, 250])
            # plt.xlabel(r'Time [$\mu s$]')
            # plt.ylabel(r'Keyhole Depth [$\mu m$]')

            # plt.show()
        # if 'depth' in key:
        #     plt.figure(dpi = 150, figsize = np.array([4,3])*1.3)

        #     plt.plot(np.arange(len(diff_results[key]))*5, diff_results[key], '--')
        #     plt.plot(np.arange(len(diff_results[key]))*5, gt_results[key], 'k-')
        #     plt.plot(np.arange(len(diff_results[key]))*5, enc_results[key], '--')
        #     plt.ylim([0, 300])
        #     frame_tick()
        #     plt.xlabel(r'Time [$\mu s$]')
        #     plt.ylabel(r'Melt Pool Depth [$\mu m$]')

        #     legend()
        #     plt.show()
    diff_amps_global.append(diff_amps)


In [ ]:
powers

In [ ]:
mean_diff = np.array(diff_amps_global).mean(axis = 0)
std_diff =  np.array(diff_amps_global).std(axis = 0)

In [ ]:
for diff_amps in diff_amps_global:
    plt.figure(figsize = [4,4])
    plt.errorbar(gt_amps, mean_diff, yerr=std_diff, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
    # plt.scatter(gt_amps, mean_diff, edgecolor = 'k' , s = 80)
    plt.plot(gt_amps, enc_amps,'o', markeredgecolor = 'k')

    plt.plot([0, 30], [0, 30], 'k-')
    # plt.plot([0, 30], [-5, 30-5], 'k--', alpha= 0.5)
    plt.plot([0, 30], [2.5, 30+2.5], 'k--', alpha= 0.5)
    plt.plot([0, 30], [-2.5, 30-2.5], 'k--', alpha= 0.5)
    plt.xlabel('')


    plt.axis('equal')
    plt.xlim([0,25])
    plt.ylim([0,25])
    frame_tick()
    legend()
    plt.show()

In [ ]:
for trial_key in diff_results_global.keys():
    print(len(diff_results_global[trial_key]))

In [ ]:
plt.figure(dpi = 150, figsize = np.array([4,3])*1.2)
plt.ylim([0,25])
# plt.grid()
diff_p = np.poly1d()
plt.errorbar(np.array(powers)/np.array(velocities), mean_diff, yerr=std_diff, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(powers)/np.array(velocities), enc_amps, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(powers)/np.array(velocities), gt_amps, fmt="ks", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.xlabel('Energy Density [J/mm]')
plt.ylabel('Keyhole Amplitude $[\mu m]$')

frame_tick()


In [ ]:
plt.figure(dpi = 150, figsize = np.array([4,3])*1.2)
plt.errorbar(np.array(powers), mean_diff, yerr=std_diff, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(powers), gt_amps, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(powers), enc_amps, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
frame_tick()


In [ ]:
plt.figure(dpi = 150, figsize = np.array([4,3])*1.2)
plt.errorbar(np.array(velocities), mean_diff, yerr=std_diff, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(velocities), gt_amps, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
plt.errorbar(np.array(velocities), enc_amps, fmt="o", capsize = 3, ecolor = 'k' ,markeredgecolor = 'k')
frame_tick()
